# AI4EAC Liquidity Stress Prediction: End-to-End Champion Pipeline

This notebook executes the complete 5-stage leak-free tournament reproduction and ensemble stacking pipeline.

### Pipeline Configuration:
- **Cross-Validation**: 1-Seed (seed=42), 10-Fold Stratified CV across all stages.
- **Validation Integrity**: Strictly nested feature screening, zero validation fold leakage, and genuine out-of-sample calibration.

### Stage Architecture:
1. **Stage 1**: Baseline Family Anchor (1-Seed 10-Fold CatBoost, XGBoost, LightGBM).
2. **Stage 2**: 1-Seed 10-Fold Domain GBDT Zoo (Nested feature screening across 5 architectures).
3. **Stage 3**: Dual TabPFN Foundation Priors (Physics & Solvency ratio representations).
4. **Stage 4**: Diversity Base Learners (MultiStrata Stratified Ensemble + Teacher-Student Distillation + Neural TabMLP).
5. **Stage 5**: 10-Fold Cross-Validated Logit-Space L2 Regularized Meta-Stacker + Single-Stage Cross-Fitted Calibration.
6. **Stage 6**: Automated Competition Invariant Audit (30k rows, probability clipping, zero NaNs, natural prevalence).
7. **Stage 7**: 20% Immutable Shadow Holdout Verification Benchmark.

In [1]:
# Setup: Clone Repository, Environment Setup, and GPU Inspection
import os, sys, time
from pathlib import Path

# 1. Automatically clone repository into /kaggle/working if src/ is missing
if not os.path.exists("src"):
    print("Cloning repository from GitHub...")
    !git clone https://github.com/Chrisolande/liquidity.git temp_repo
    !cp -r temp_repo/* .
    !rm -rf temp_repo
    print("Repository cloned successfully.")
else:
    print("src/ directory detected in working tree.")

# 2. Install required dependencies
!pip install -q feature-engine tabpfn

# 3. Ensure repository root is in python path
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch
print(f"Working Directory: {repo_root}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from src.config import SEED, N_SPLITS, get_default_dataset_paths
print(f"Global Seed: {SEED} | Folds: {N_SPLITS}")
tr_p, te_p = get_default_dataset_paths()
print(f"Dataset Paths: Train={tr_p}, Test={te_p}")

# ==============================================================================
# PIPELINE EXECUTION TOGGLES
# ==============================================================================
SINGLE_SEED_MODE = True  # Set to False to run full multi-seed ensemble (42, 2026)
ENABLE_STAGE3 = False    # Set to True to enable heavy TabPFN priors (~2h)

from src.config import set_seed_mode, set_stage3_mode, ACTIVE_SEEDS
RUN_SEEDS = set_seed_mode(single_seed=SINGLE_SEED_MODE)
STAGE3_ACTIVE = set_stage3_mode(enable=ENABLE_STAGE3)
print(f"Seed Mode: {'SINGLE-SEED' if SINGLE_SEED_MODE else 'MULTI-SEED'} -> Active Seeds: {RUN_SEEDS}")
print(f"Stage 3 (TabPFN): {'ENABLED' if STAGE3_ACTIVE else 'DISABLED (Fast Iteration)'}")


Cloning repository from GitHub...
Cloning into 'temp_repo'...
remote: Enumerating objects: 463, done.
remote: Counting objects: 100% (463/463), done.
remote: Compressing objects: 100% (256/256), done.
remote: Total 463 (delta 221), reused 422 (delta 180), pack-reused 0 (from 0)
Receiving objects: 100% (463/463), 23.57 MiB | 34.00 MiB/s, done.
Resolving deltas: 100% (221/221), done.
Repository cloned successfully.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 6.1 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 827.5/827.5 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 28.5 MB/s eta 0:00:00
Working Directory: /kaggle/working
CUDA Available: True
GPU: Tesla T4
Global Seed: 42 | Folds: 10
Dataset Paths: Train=/kaggle/input/datasets/chrisolande2/zindi-competition/Train.csv, Test=/kaggle/input/datasets/chrisolande2/zindi-competition/Test.csv
Seed Mode: SINGLE-SEED -> Active Seeds: (42,)
Stage 3 (TabPFN): DISABLED (Fast 

## SECTION A: Clean Leak-Free Validation & Stabilized Submission
Executes 10-fold CV with strictly fold-local feature screening, fold-local target encoding,
zero pseudo-labeling, zero temperature sharpening, zero test prevalence hacking, and regularized blending.
Generates `submissions/submission_clean.csv` and `artifacts/experiment.json` with `LEAKAGE CHECK: PASS`.

In [ ]:
import time
from src.clean_pipeline import run_clean_pipeline

t0 = time.time()
clean_manifest = run_clean_pipeline(
    seeds=RUN_SEEDS,
    n_splits=10,
    k_top_features=60,
    run_tabpfn=False,
    output_dir="artifacts",
    sub_dir="submissions",
)
print(f"\nClean Pipeline complete in {(time.time() - t0)/60:.2f} min | Composite Score: {clean_manifest['oof_metrics']['composite_score']:.5f}")


In [ ]:
import pandas as pd
sub_clean = pd.read_csv("submissions/submission_clean.csv")
print(f"Clean Submission Shape: {sub_clean.shape}")
print(f"Range: [{sub_clean["Target"].min():.4f}, {sub_clean["Target"].max():.4f}] | Mean: {sub_clean["Target"].mean():.4f}")
print(sub_clean.head())


## Stage 1: Tournament Baseline Anchor Reproduction & Hill Climbing
Trains 1-seed (seed=42) 10-fold CatBoost, XGBoost, and HistGradientBoosting on the selected features, optimizes weights via SLSQP, applies calibration, and runs stepwise hill climbing.

In [ ]:
from src.stages.stage1_reproduce import run_stage1
import time
t0 = time.time()
s1_score = run_stage1(seeds=RUN_SEEDS)
print(f"Stage 1 complete in {(time.time() - t0)/60:.2f} min | Composite Score: {s1_score:.5f}")


Stage 1: Baseline and hill climbing pipeline
Stage 1A: Baseline 1-seed ensemble (10-fold CV)
Step 1: Engineering comprehensive feature set
Loaded cached features from checkpoints/feature_cache (906 features)
Step 2: Running feature_engine selection (targeting ~60 features)
Feature selection: initial count = 904 features
feature_engine.DropConstantFeatures: dropped 7 features
feature_engine.DropDuplicateFeatures: dropped 145 features
feature_engine.SmartCorrelatedSelection (r>0.98): dropped 62 features
Feature selection complete: 79 features selected (out of 904)
Step 3: Multi-seed cross-validation — 10-fold CV, seeds=[42], features=79 (74 numeric + 5 categorical)
Seed 1/1 (seed=42)
  [seed=42] Training model: catboost
    Fold 1 | LogLoss: 0.2267 | AUC: 0.9253 | Comp: 0.7415
    Fold 2 | LogLoss: 0.2187 | AUC: 0.9305 | Comp: 0.7516
    Fold 3 | LogLoss: 0.2246 | AUC: 0.9274 | Comp: 0.7445
    Fold 4 | LogLoss: 0.2345 | AUC: 0.9155 | Comp: 0.7297
    Fold 5 | LogLoss: 0.2355 | AUC: 0.91

## Stage 2: 10-Fold Multi-Model Domain GBDT Zoo
Executes CatBoost, XGBoost, and LightGBM with leak-free fold-nested feature screening across specialized domain subsets (solvency ratios, balance collapses, and triage columns).

In [ ]:
from src.stages.stage2_gbdt_zoo import run_stage2
import time
t0 = time.time()
s2_score = run_stage2(seeds=RUN_SEEDS, n_splits=10, include_anchor=True)
print(f"Stage 2 complete in {(time.time() - t0)/60:.2f} min | Composite Score: {s2_score:.5f}")


## Stage 3: Dual TabPFN Foundation Priors
Extracts non-tree foundation model representations on GPU using RankGauss Quantile Transformation on the physical solvency ratio views.

In [ ]:
import time
from src.stages.stage3_tabpfn_priors import run_stage3

t0 = time.time()
if ENABLE_STAGE3:
    print("Running Stage 3 Dual TabPFN Foundation Priors...")
    s3_score = run_stage3(n_splits=10, enabled=True)
    print(f"Stage 3 complete in {(time.time() - t0)/60:.2f} min | Composite Score: {s3_score:.5f}")
else:
    print(">>> Stage 3 (TabPFN) is DISABLED via toggle to save time.")
    s3_score = run_stage3(n_splits=10, enabled=False)


## Stage 4: Diversity Pipeline (MultiStrata, Distillation Students, Neural TabMLP)
Executes:
1. MultiStrata 5-Fold Iterative Stratification Ensemble (906 features).
2. 10-Fold Teacher-Student Distillation (CatBoost & XGBoost) on clean soft probability targets.
3. PyTorch Neural TabMLP with residual connections and bounded logit scaling.

In [ ]:
import time
from src.stages.stage4_diversity import run_stage4

t0 = time.time()
s4_score = run_stage4()
print(f"Stage 4 complete in {(time.time() - t0)/60:.2f} min | Composite Score: {s4_score:.5f}")


## Stage 5: 10-Fold Cross-Validated Logit-Space L2 Meta-Stacker
Solves analytical L-BFGS-B regularized regression in logit space across all candidate model streams, eliminates in-sample hill-climbing, and applies single-stage cross-fitted Platt calibration.

In [ ]:
import time
from src.stages.stage5_meta_stacker import run_stage5

t0 = time.time()
final_champion_score = run_stage5()
print(f"Stage 5 complete: Champion Score = {final_champion_score:.5f}")


## Stage 6: Final Verification & Invariants Audit
Checks all 5 hard competition invariants: exact row counts (30,000), column names, zero NaNs, probability bounding [0.002, 0.998], and benchmark correlation.

In [ ]:
import os
import pandas as pd
from src.stages.audit import run_audit

# Run automated audit
audit_passed = run_audit()
print(f"Audit Passed: {audit_passed}")

# Inspect final submission preview
sub_file = "submissions/submission_stage5_final.csv"
if os.path.exists(sub_file):
    df_sub = pd.read_csv(sub_file)
    print(f"Submission Shape: {df_sub.shape}")
    print(f"Mean Predicted Probability: {df_sub['Target'].mean():.5f}")
    print(f"5th Percentile: {df_sub['Target'].quantile(0.05):.5f} | 95th Percentile: {df_sub['Target'].quantile(0.95):.5f}")
    print("First 10 Rows:")
    print(df_sub.head(10))
else:
    print(f"{sub_file} not found yet.")

## Stage 7: 20% Immutable Shadow Holdout Verification Benchmark
Partitions an untouched 20% shadow holdout (never seen by feature screening, training, or stacking) to empirically prove that the clean leak-free pipeline outperforms the legacy optimistic pipeline out-of-sample.

In [ ]:
from experiments.run_shadow_holdout_benchmark import run_benchmark

# Execute 20% Shadow Holdout Benchmark
benchmark_passed = run_benchmark()
print(f"Shadow Holdout Verification Success: {benchmark_passed}")
